# R01: Scratch lstm, aspect attention

No pretrained weights. Train-only BPE. Official dev RMSE_VA.
Two concurrent Kaggle sessions maximum.


In [ ]:
RUN = "R01"
ENCODER = "lstm"
POOLING = "attention"
LAYERS = 2
LEARNING_RATE = 0.002


In [ ]:
import torch
print("torch:",torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required for this experiment; select accelerator before Save & Run.")
print("GPU:",torch.cuda.get_device_name(0))


## Đọc dữ liệu đã upload
Cần attach dataset private chứa thư mục `code/`, `data/` và scorer BTC.


In [ ]:
from pathlib import Path
import json, subprocess, sys

source_files = list(Path("/kaggle/input").rglob("train_scratch.py"))
if len(source_files) != 1:
    raise RuntimeError("Attach exactly one prepared DimABSA training bundle.")
code_dir = source_files[0].parent
data_dir = code_dir.parent / "data"
output = Path("/kaggle/working") / RUN
output.mkdir(parents=True, exist_ok=True)
print({"code": str(code_dir), "data": str(data_dir), "output": str(output)})


## Train và giữ log
Hai nhánh: BiLSTM + attention; Transformer pre-norm + attention. Cùng split, BPE policy, seed và budget.


In [ ]:
command = [sys.executable, str(code_dir / "train_scratch.py"), "--data", str(data_dir), "--out", str(output / "model"), "--run", RUN, "--pooling", POOLING, "--encoder", ENCODER, "--layers", str(LAYERS), "--lr", str(LEARNING_RATE)]
with (output / "run.log").open("w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f"Training failed: exit code {exit_code}")


## Chấm lại bằng scorer BTC
Giữ nguyên ID, aspect, thứ tự và định dạng VA hai chữ số.


In [ ]:
command = [sys.executable, str(code_dir.parent / "metrics_subtask_1_2_3.py"), "-t", "1", "-g", str(data_dir / "eng_restaurant_dev_task1.jsonl"), "-p", str(output / "model/dev_predictions.jsonl")]
result = subprocess.run(command, capture_output=True, text=True, check=True)
(output / "official_scorer.log").write_text(result.stdout)
print(result.stdout)
report = json.loads((output / "model/report.json").read_text())
print(json.dumps(report, indent=2))
